<a href="https://colab.research.google.com/github/suyogy1-bot/Machine_Learning/blob/main/Fake_Job_Post_Detector_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [ ]:
# Install dependencies as needed:
%pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "fake_job_postings.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "shivamb/real-or-fake-fake-jobposting-prediction",
  file_path,
)



zsh:1: no matches found: kagglehub[pandas-datasets]
Note: you may need to restart the kernel to use updated packages.


/var/folders/tj/5m337bwx0gndn28_h0bqpcz00000gn/T/ipykernel_13331/2780137391.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


## Load data
### 2. Combine the text columns into one column called "text"

In [ ]:
df["text"] = (
    df["title"].fillna("") + " "
    + df["company_profile"].fillna("") + " "
    + df["description"].fillna("") + " "
    + df["requirements"].fillna("")
)

df.head()

df.drop(columns = ['title', 'company_profile', 'description', 'requirements', 'job_id', 'location','department'
                   , 'benefits', 'salary_range', 'function', 'industry'], inplace = True)


In [ ]:
df.head(5)
df.isnull().sum()


telecommuting             0
has_company_logo          0
has_questions             0
employment_type        3471
required_experience    7050
required_education     8105
fraudulent                0
text                      0
dtype: int64

### 3. Fill missing values in categorical columns

In [ ]:
cat_col = ['employment_type', 'required_experience', 'required_education']
for col in cat_col:
    df[col] = df[col].fillna('Unknown')

num_col = ['telecommuting', 'has_company_logo', 'has_questions']

### 4. Train / test split (stratify because fraud is rare)

In [ ]:
X = df[['text'] + cat_col + num_col]
y = df['fraudulent']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 5. ColumnTransformer: different processing for different columns

In [ ]:
preprocessor = ColumnTransformer(transformers = [
    ('text', CountVectorizer(), 'text'),
    ('cat', OneHotEncoder(handle_unknown ='ignore'), cat_col),
    ('num', 'passthrough', num_col)
])



### 6. Pipeline: preprocessing + model

In [ ]:
model = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter = 1000, class_weight = 'balanced'))
])
model.fit(x_train, y_train)


model_2 = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter = 1000, class_weight = 'balanced'))
])
model.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text', CountVectorizer(),
                                                  'text'),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['employment_type',
                                                   'required_experience',
                                                   'required_education']),
                                                 ('num', 'passthrough',
                                                  ['telecommuting',
                                                   'has_company_logo',
                                                   'has_questions'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

### 7. Evaluate

In [ ]:
y_pred = model.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

joblib.dump(model, 'fake_job_post_detector_model.pkl')
print("Model saved as 'fake_job_post_detector_model.pkl'")

Accuracy: 0.9787472035794184
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3395
           1       0.77      0.83      0.80       181

    accuracy                           0.98      3576
   macro avg       0.88      0.91      0.89      3576
weighted avg       0.98      0.98      0.98      3576

Model saved as 'fake_job_post_detector_model.pkl'


### 8. Save

In [ ]:
pip show scikit-learn

Name: scikit-learn
Version: 1.4.2
Summary: A set of python modules for machine learning and data mining
Home-page: https://scikit-learn.org
Author: 
Author-email: 
License: new BSD
Location: /opt/anaconda3/lib/python3.12/site-packages
Requires: joblib, numpy, scipy, threadpoolctl
Required-by: imbalanced-learn
Note: you may need to restart the kernel to use updated packages.
